In [ ]:
# Cài đặt PyTorch chuẩn GPU và các thư viện cần thiết
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install timm opencv-python pillow tqdm onnx onnxruntime

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 123.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os
import cv2
import glob
from tqdm import tqdm

# ==========================================
# 1. KẾT NỐI DRIVE & CHUẨN BỊ THƯ MỤC
# ==========================================
drive.mount('/content/drive')

# ⚠️ LƯU Ý QUAN TRỌNG: Sửa lại tên file "FaceForensics.zip" nếu file trên Drive của bạn có tên khác (VD: FFPP.zip)
zip_path = "/content/drive/MyDrive/DoAn_Nhom4/FaceForensics.zip"

extract_path = "/content/ffpp_raw"
DATASET_ROOT = '/content/dataset_ffpp'
REAL_OUT_DIR = os.path.join(DATASET_ROOT, 'real')
FAKE_OUT_DIR = os.path.join(DATASET_ROOT, 'fake')

os.makedirs(extract_path, exist_ok=True)
os.makedirs(REAL_OUT_DIR, exist_ok=True)
os.makedirs(FAKE_OUT_DIR, exist_ok=True)

# ==========================================
# 2. COPY VÀ GIẢI NÉN DỮ LIỆU
# ==========================================
print("📦 Đang copy file nén từ Drive vào Colab (Vui lòng đợi)...")
# Copy file từ Drive sang bộ nhớ tạm của Colab để giải nén nhanh hơn
!cp "{zip_path}" "/content/"

print("🗜️ Đang giải nén dữ liệu...")
# Lưu ý: Sửa lại tên file trong ngoặc kép nếu ở trên bạn đã đổi tên
!unzip -q "/content/FaceForensics.zip" -d "{extract_path}"

# Xóa file nén đi để tránh bị nổ ổ cứng Colab (Disk Full)
!rm "/content/FaceForensics.zip"
print("✅ Giải nén hoàn tất! Đã dọn dẹp file rác.")

# ==========================================
# 3. TRÍCH XUẤT KHUNG HÌNH (PHIÊN BẢN C23)
# ==========================================
FRAMES_PER_VIDEO = 8

def extract_frames(video_path, output_dir, label_prefix):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return

    intervals = [int(i * total_frames / FRAMES_PER_VIDEO) for i in range(FRAMES_PER_VIDEO)]
    video_name = os.path.basename(video_path).split('.')[0]

    # Lấy tên thư mục ngay sát file mp4 (VD: 'original', 'Deepfakes', 'Face2Face')
    method_name = os.path.basename(os.path.dirname(video_path))

    for i, frame_idx in enumerate(intervals):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            out_path = os.path.join(output_dir, f"{label_prefix}_{method_name}_{video_name}_frame_{i}.jpg")
            cv2.imwrite(out_path, frame)
    cap.release()

print("\n⚙️ BẮT ĐẦU CẮT ẢNH TỪ VIDEO...")

# 1. Tìm Video THẬT (Chỉ vào thư mục 'original')
real_videos = glob.glob(f"{extract_path}/**/original/**/*.mp4", recursive=True)

# 2. Tìm Video GIẢ (Gom từ 5 thư mục phương pháp, bỏ qua csv)
fake_methods = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures', 'FaceShifter']
fake_videos = []
for method in fake_methods:
    fake_videos.extend(glob.glob(f"{extract_path}/**/{method}/**/*.mp4", recursive=True))

print(f"🟢 Tìm thấy {len(real_videos)} video THẬT.")
print(f"🔴 Tìm thấy {len(fake_videos)} video GIẢ.")

# Bắt đầu chạy thanh tiến trình
for v in tqdm(real_videos, desc="Xử lý Video THẬT"):
    extract_frames(v, REAL_OUT_DIR, "real")

for v in tqdm(fake_videos, desc="Xử lý Video GIẢ"):
    extract_frames(v, FAKE_OUT_DIR, "fake")

print(f"\n✅ HOÀN TẤT TẤT CẢ! Đã có {len(os.listdir(REAL_OUT_DIR))} ảnh Thật và {len(os.listdir(FAKE_OUT_DIR))} ảnh Giả.")

Mounted at /content/drive
📦 Đang copy file nén từ Drive vào Colab (Vui lòng đợi)...
🗜️ Đang giải nén dữ liệu...
✅ Giải nén hoàn tất! Đã dọn dẹp file rác.

⚙️ BẮT ĐẦU CẮT ẢNH TỪ VIDEO...
🟢 Tìm thấy 1000 video THẬT.
🔴 Tìm thấy 5000 video GIẢ.


Xử lý Video GIẢ: 100%|██████████| 5000/5000 [1:00:07<00:00,  1.39it/s]


✅ HOÀN TẤT TẤT CẢ! Đã có 8000 ảnh Thật và 40000 ảnh Giả.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Trái tim hệ thống: {device}")

swin_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
rppg_transform = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])

class RealDeepfakeDataset(Dataset):
    def __init__(self, root_dir, visual_transform=None, bio_transform=None):
        self.visual_transform = visual_transform
        self.bio_transform = bio_transform
        self.image_paths, self.labels = [], []

        for label, folder in enumerate(['real', 'fake']):
            folder_path = os.path.join(root_dir, folder)
            if os.path.exists(folder_path):
                for f in os.listdir(folder_path):
                    if f.endswith(('.png', '.jpg', '.jpeg')):
                        self.image_paths.append(os.path.join(folder_path, f))
                        self.labels.append(label)

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img_cv2 = cv2.imread(self.image_paths[idx])
        if img_cv2 is None: img_cv2 = np.zeros((224, 224, 3), dtype=np.uint8)
        img_pil = Image.fromarray(cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB))
        return self.visual_transform(img_pil), self.bio_transform(img_pil), torch.tensor(self.labels[idx], dtype=torch.float32)

try:
    dataset = RealDeepfakeDataset(DATASET_ROOT, swin_transform, rppg_transform)
    dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)
    print("✅ Đã nạp dữ liệu vào DataLoader thành công!")
except Exception as e:
    print(f"❌ Lỗi nạp DataLoader: {e}")

🖥️ Trái tim hệ thống: cuda
✅ Đã nạp dữ liệu vào DataLoader thành công!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import timm
import os

# ==========================================
# 1. KIẾN TRÚC MÔ HÌNH NÂNG CẤP (CÓ CROSS-ATTENTION)
# ==========================================
class DeepfakeFusionModel(nn.Module):
    def __init__(self):
        super(DeepfakeFusionModel, self).__init__()
        self.swin = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)

        self.rppg_net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1))
        )

        # Lớp Attention: Tự động đánh giá trọng số giữa Swin và rPPG
        self.attention_weights = nn.Sequential(
            nn.Linear(768 + 64, 768 + 64),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(768 + 64, 128),
            nn.BatchNorm1d(128), # Giúp mạng ổn định hơn khi dùng Attention
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, image_input, rppg_input):
        visual_features = self.swin(image_input)
        bio_features = torch.flatten(self.rppg_net(rppg_input), 1)

        combined_features = torch.cat((visual_features, bio_features), dim=1)
        attention_scores = self.attention_weights(combined_features)

        # Nhân đặc trưng với điểm Attention (Lọc nhiễu)
        attended_features = combined_features * attention_scores

        return self.classifier(attended_features)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepfakeFusionModel().to(device)
criterion = nn.BCELoss()

# ==========================================
# 2. XỬ LÝ MẤT CÂN BẰNG DỮ LIỆU BẰNG SAMPLER
# ==========================================
# (Giả định biến 'dataset' của FF++ đã được nạp ở Ô Code trước đó)
# Đếm số lượng ảnh từng lớp
labels_list = dataset.labels
class_counts = [labels_list.count(0), labels_list.count(1)]
print(f"📊 Thống kê dữ liệu: {class_counts[0]} Real | {class_counts[1]} Fake")

# Tính toán trọng số: Lớp nào ít ảnh hơn sẽ có trọng số cao hơn
num_samples = sum(class_counts)
class_weights = [num_samples / class_counts[i] for i in range(len(class_counts))]
sample_weights = [class_weights[labels_list[i]] for i in range(len(labels_list))]

# Khởi tạo Sampler thần thánh
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=num_samples, replacement=True)

# LƯU Ý QUAN TRỌNG: Khi dùng Sampler, BẮT BUỘC phải tắt shuffle=False
dataloader_balanced = DataLoader(dataset, batch_size=16, sampler=sampler, num_workers=2)
print("⚖️ Đã cấu hình bộ cân bằng dữ liệu WeightedRandomSampler!")

# ==========================================
# 3. VÒNG LẶP HUẤN LUYỆN (FINE-TUNING LẦN 2)
# ==========================================
# Load lại trí nhớ mạnh nhất của Celeb-DF (Phase 1) để rèn luyện tiếp
old_weights = '/content/drive/MyDrive/DoAn_Nhom4/weights_phase1_celeb/fusion_model_celeb_epoch_30.pth'
try:
    # strict=False vì mô hình mới có thêm lớp Attention, nó sẽ bỏ qua việc thiếu trọng số lớp này và chỉ nạp Swin + rPPG
    model.load_state_dict(torch.load(old_weights, map_location=device), strict=False)
    print("✅ Đã kế thừa bộ não Phase 1. Các lớp Attention mới sẽ được học từ đầu!")
except Exception as e:
    print(f"⚠️ Cảnh báo Load Model: {e}")

save_path = '/content/drive/MyDrive/DoAn_Nhom4/weights_ffpp_v2/'
os.makedirs(save_path, exist_ok=True)

# Tốc độ học nhỏ vì đang fine-tuning
optimizer = optim.AdamW(model.parameters(), lr=1e-5)
epochs = 20

print(f"\n🚀 BẮT ĐẦU HUẤN LUYỆN BẢN NÂNG CẤP ({epochs} EPOCHS) 🚀")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, rppg_maps, labels) in enumerate(dataloader_balanced):
        images, rppg_maps, labels = images.to(device), rppg_maps.to(device), labels.float().to(device)

        optimizer.zero_grad()
        outputs = model(images, rppg_maps).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"🔄 Epoch [{epoch+1}/{epochs}] - Batch [{batch_idx}/{len(dataloader_balanced)}] - Loss: {loss.item():.4f}")

    print(f"⭐ TỔNG KẾT EPOCH {epoch+1}: Trung bình Loss = {running_loss / len(dataloader_balanced):.4f}")

    checkpoint_file = f"{save_path}fusion_model_ffpp_v2_epoch_{epoch+1}.pth"
    torch.save(model.state_dict(), checkpoint_file)
    torch.cuda.empty_cache()

print("🎉 HOÀN TẤT HUẤN LUYỆN BẢN NÂNG CẤP!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

📊 Thống kê dữ liệu: 8000 Real | 40000 Fake
⚖️ Đã cấu hình bộ cân bằng dữ liệu WeightedRandomSampler!
⚠️ Cảnh báo Load Model: [Errno 2] No such file or directory: '/content/drive/MyDrive/DoAn_Nhom4/weights_phase1_celeb/fusion_model_celeb_epoch_30.pth'

🚀 BẮT ĐẦU HUẤN LUYỆN BẢN NÂNG CẤP (20 EPOCHS) 🚀
🔄 Epoch [1/20] - Batch [0/3000] - Loss: 0.7159
🔄 Epoch [1/20] - Batch [50/3000] - Loss: 0.8262
🔄 Epoch [1/20] - Batch [100/3000] - Loss: 0.8492
🔄 Epoch [1/20] - Batch [150/3000] - Loss: 0.7816
🔄 Epoch [1/20] - Batch [200/3000] - Loss: 0.7657
🔄 Epoch [1/20] - Batch [250/3000] - Loss: 0.7778
🔄 Epoch [1/20] - Batch [300/3000] - Loss: 0.6466
🔄 Epoch [1/20] - Batch [350/3000] - Loss: 0.8124
🔄 Epoch [1/20] - Batch [400/3000] - Loss: 0.8146
🔄 Epoch [1/20] - Batch [450/3000] - Loss: 0.7805
🔄 Epoch [1/20] - Batch [500/3000] - Loss: 0.6552
🔄 Epoch [1/20] - Batch [550/3000] - Loss: 0.7494
🔄 Epoch [1/20] - Batch [600/3000] - Loss: 0.9055
🔄 Epoch [1/20] - Batch [650/3000] - Loss: 0.6495
🔄 Epoch [1/20] - 

In [5]:
import torch.optim as optim

# 1. LOAD TRÍ NHỚ CŨ (KẾ THỪA TỪ CELEB-DF)
old_weights = '/content/drive/MyDrive/DoAn_Nhom4/weights/fusion_model_epoch_30.pth'
try:
    model.load_state_dict(torch.load(old_weights, map_location=device))
    print("✅ Đã kế thừa thành công bộ não từ Giai đoạn 1!")
except Exception as e:
    print(f"⚠️ Không tìm thấy file trọng số cũ, mô hình sẽ học lại từ đầu. Lỗi: {e}")

# 2. CẤU HÌNH HUẤN LUYỆN
save_path = '/content/drive/MyDrive/DoAn_Nhom4/weights_ffpp/'
os.makedirs(save_path, exist_ok=True)

# LR=1e-5 để không quên kiến thức cũ
optimizer = optim.AdamW(model.parameters(), lr=1e-5)
epochs = 20

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN GIAI ĐOẠN 2 ({epochs} EPOCHS) 🚀")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, rppg_maps, labels) in enumerate(dataloader):
        images, rppg_maps, labels = images.to(device), rppg_maps.to(device), labels.float().to(device)

        optimizer.zero_grad()
        outputs = model(images, rppg_maps).squeeze(1) # Chống lỗi lô cuối cùng
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"🔄 Epoch [{epoch+1}/{epochs}] - Batch [{batch_idx}/{len(dataloader)}] - Loss: {loss.item():.4f}")

    print(f"⭐ TỔNG KẾT EPOCH {epoch+1}: Trung bình Loss = {running_loss / len(dataloader):.4f}")

    checkpoint_file = f"{save_path}fusion_model_ffpp_epoch_{epoch+1}.pth"
    torch.save(model.state_dict(), checkpoint_file)
    print(f"✅ Đã lưu trọng số: {checkpoint_file}")

    torch.cuda.empty_cache()
print("🎉 HOÀN TẤT HUẤN LUYỆN GIAI ĐOẠN 2!")

⚠️ Không tìm thấy file trọng số cũ, mô hình sẽ học lại từ đầu. Lỗi: Error(s) in loading state_dict for DeepfakeFusionModel:
	Missing key(s) in state_dict: "attention_weights.0.weight", "attention_weights.0.bias", "classifier.1.weight", "classifier.1.bias", "classifier.1.running_mean", "classifier.1.running_var", "classifier.4.weight", "classifier.4.bias". 
	Unexpected key(s) in state_dict: "classifier.3.weight", "classifier.3.bias". 
🚀 BẮT ĐẦU HUẤN LUYỆN GIAI ĐOẠN 2 (20 EPOCHS) 🚀
🔄 Epoch [1/20] - Batch [0/3000] - Loss: 0.7839
🔄 Epoch [1/20] - Batch [50/3000] - Loss: 0.6496
🔄 Epoch [1/20] - Batch [100/3000] - Loss: 0.8789
🔄 Epoch [1/20] - Batch [150/3000] - Loss: 0.7843
🔄 Epoch [1/20] - Batch [200/3000] - Loss: 0.7496
🔄 Epoch [1/20] - Batch [250/3000] - Loss: 0.6600
🔄 Epoch [1/20] - Batch [300/3000] - Loss: 0.7289
🔄 Epoch [1/20] - Batch [350/3000] - Loss: 0.5994
🔄 Epoch [1/20] - Batch [400/3000] - Loss: 0.6671
🔄 Epoch [1/20] - Batch [450/3000] - Loss: 0.6973
🔄 Epoch [1/20] - Batch [500/

KeyboardInterrupt: 

In [ ]:
print("⚙️ Đang xuất file .onnx chuẩn bị tích hợp Backend...")
model.eval()

# Tạo input giả lập để xuất đồ thị
dummy_image = torch.randn(1, 3, 224, 224).to(device)
dummy_rppg = torch.randn(1, 3, 64, 64).to(device)
onnx_path = f"{save_path}deepfake_fusion_final_web.onnx"

torch.onnx.export(
    model, (dummy_image, dummy_rppg), onnx_path,
    export_params=True, opset_version=14,
    input_names=['image_input', 'rppg_input'], output_names=['output'],
    dynamic_axes={'image_input': {0: 'batch_size'}, 'rppg_input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f"🚀 Đã xuất xong file ONNX siêu nhẹ tại: {onnx_path}")

Đang xuất mô hình ra chuẩn Web (ONNX)...


ModuleNotFoundError: No module named 'onnxscript'